# 昇腾芯片

昇腾（Ascend）是华为面向人工智能计算设计的 AI 处理器系列，也常被称为 NPU（Neural Processing Unit）。在 AI 计算场景中，NPU 会针对神经网络中的矩阵计算、向量计算和数据搬运等典型负载进行专门优化，以提升训练和推理任务的执行效率。

在 PyPTO 学习中，理解昇腾芯片并不是为了直接编写硬件指令，而是为了理解为什么 PyPTO 会强调 Tile、内存层次、并行粒度和数据类型。这些概念都和芯片的执行方式密切相关。

## 1. 面向 AI 计算的硬件特点

昇腾芯片围绕 AI 计算场景进行了专门设计，典型特点包括：

- **专用计算单元**：通过 AI Core 等计算单元加速矩阵、向量和张量相关计算。
- **多核并行能力**：多个 AI Core 可以并行处理不同任务或不同数据块，提高整体吞吐。
- **多级内存层次**：通过片上缓存和片外内存配合，降低数据访问成本。
- **多精度支持**：支持训练、推理和量化场景中常见的多种数据类型。

这些硬件特点决定了高性能算子开发不能只关注数学公式，还需要关注数据如何切分、如何搬运、如何复用，以及如何映射到多个计算核心上。

## 2. AI Core 与并行执行

AI Core 是昇腾芯片执行 AI 计算的重要单元。实际运行时，一个算子通常会被拆分成多个可以并行执行的任务，分配到不同 AI Core 上处理。

在这种模式下，开发者需要理解两个层面的并行：

1. **计算并行**：多个 AI Core 同时执行计算，提高整体吞吐。
2. **数据并行**：大 Tensor 被切分成多个 Tile，不同核心处理不同数据块。

昇腾 AI 处理器支持 MPMD（Multiple Program Multiple Data）执行模式，即不同核心可以执行不同程序，并处理不同数据。对于复杂融合算子或大模型组件来说，这种模式为更灵活的调度和并行优化提供了基础。

## 3. 内存层次与 Tile 思想

AI 计算的性能不仅取决于算力，也取决于数据是否能高效地到达计算单元。昇腾芯片采用多级内存层次，例如 L0 Buffer、L1 Buffer、Unified Buffer、Global Memory 以及片外 HBM 等。

不同层级的内存容量、带宽和访问延迟不同。通常越靠近计算单元，访问越快但容量越小；越远离计算单元，容量越大但访问成本越高。因此，高性能算子需要尽量让数据在靠近计算单元的缓存中复用，减少频繁访问外部内存。

这也是 PyPTO 强调 Tile 编程模型的重要原因：

- 大 Tensor 会被切分成适合片上缓存和并行计算的数据块。
- 每个 Tile 可以在有限的片上内存中完成搬运、计算和复用。
- 合理的 Tile 形状会影响并行粒度、内存访问效率和整体性能。

理解内存层次，有助于后续理解 PyPTO 中 TileShape、数据搬运和算子融合等概念。

## 4. 精度支持

昇腾芯片支持多种数据类型，以适配不同训练和推理场景。常见类型包括 FP32、FP16、BF16、INT8、INT4 等，也包括部分面向低精度计算的格式，如 FP8E4M3、FP8E5M2、FP8E8M0。

不同精度通常对应不同的性能、显存占用和数值稳定性：

- 高精度类型更有利于数值稳定，但计算和存储成本较高。
- 低精度类型可以降低内存占用、提升吞吐，常用于推理或量化场景。
- 大模型训练和推理中，常会结合多种精度来平衡性能与精度。

因此，在 PyPTO 算子开发中，数据类型不仅是接口参数，也会影响计算路径、内存占用和性能优化策略。

## 5. 软件生态与开发支持

昇腾芯片并不是孤立使用的硬件设备，而是配套完整软件生态共同工作：

1. **CANN 软件栈**：提供驱动、固件、Toolkit、Ops 包、编译链和运行时环境。
2. **PyPTO 框架**：面向昇腾高性能算子开发，提供 Tensor 和 Tile 级别的编程抽象。
3. **框架适配层**：支持 PyTorch、TensorFlow 等主流深度学习框架在昇腾硬件上运行，例如 PyTorch 可通过 `torch_npu` 扩展接入。
4. **底层开发工具**：包括 Ascend C、性能分析工具、精度验证工具等。

从应用角度看，昇腾芯片可用于大模型训练、推理服务、边缘计算和云端 AI 服务等场景。对于本教程而言，重点是理解这些硬件能力如何影响 PyPTO 的算子开发方式。

## 6. 本节小结

昇腾芯片提供 AI 计算的硬件基础。AI Core、多级内存、多精度支持和 MPMD 执行模式共同决定了算子开发需要关注并行、数据搬运、内存复用和数据类型选择。

下一节将介绍 PyPTO。理解了芯片的这些基本特征后，就更容易明白为什么 PyPTO 要围绕 Tensor、Tile、多层级计算图和自动代码生成来设计。